In [0]:
import pandas as pd #import pandas library
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.getOrCreate()

df=pd.read_csv("https://raw.githubusercontent.com/Bhevendra/ML-Datasets/refs/heads/main/retail_data/customers.csv")# read csv file

# Convert Pandas DataFrame to Spark DataFrame
df = spark.createDataFrame(df)


In [0]:
df

In [0]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.getOrCreate()

# Convert Pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

# Write to Delta table
spark_df.write.format("delta").mode("overwrite").saveAsTable("batch_1.data.customers")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, explode, schema_of_json, lit

spark = SparkSession.builder.getOrCreate()

# مثال: load data (change as needed)
df = spark.read.table("batch_1.data.customers")

# ---------------------------
# SCHEMA INFERENCE
# ---------------------------
sample_ordered = df.select("ordered_products") \
    .filter(col("ordered_products").isNotNull()) \
    .first()[0]

ordered_schema = schema_of_json(lit(sample_ordered))

# ---------------------------
# PARSE + EXPLODE
# ---------------------------
df_parsed = df.withColumn(
    "ordered_products_json",
    from_json(col("ordered_products"), ordered_schema)
)

df_exploded = df_parsed.withColumn(
    "product",
    explode(col("ordered_products_json"))
)

# ---------------------------
# PROMO SCHEMA
# ---------------------------
promo_sample = df_exploded.select("product.promotion_info") \
    .filter(col("product.promotion_info").isNotNull()) \
    .first()[0]

promo_schema = schema_of_json(lit(promo_sample))

df_fixed = df_exploded.withColumn(
    "promo",
    from_json(col("product.promotion_info"), promo_schema)
)

# ---------------------------
# FINAL FLATTEN
# ---------------------------
df_flat = df_fixed.select(
    "customer_id",
    "customer_name",
    "order_number",
    "order_datetime",
    "number_of_line_items",

    col("product.curr").alias("curr"),
    col("product.id").alias("product_id"),
    col("product.name").alias("product_name"),
    col("product.price").cast("int").alias("price"),
    col("product.qty").cast("int").alias("qty"),
    col("product.unit").alias("unit"),

    col("promo.promo_disc"),
    col("promo.promo_id"),
    col("promo.promo_item"),
    col("promo.promo_qty")
)

display(df_flat)